# Etapa 2 — Análise Exploratória e Limpeza

**ImunizaData** — cobertura vacinal por município (SUS).

Este notebook lê o dataset já limpo e cruzado (`refined/cobertura_vacinal`,
gerado por `src/cleaning/build_coverage.py`) e faz a análise exploratória:
distribuição da métrica de cobertura, outliers, e o cruzamento com
população.

> Antes de rodar este notebook: `docker-compose up -d`, depois
> `python -m src.ingestion.download_ibge --ano 2024`,
> `python -m src.ingestion.download_pni --ano 2025`,
> `python -m src.cleaning.clean_ibge --ano 2024`,
> `python -m src.cleaning.clean_pni --ano 2025`,
> `python -m src.cleaning.build_coverage --ano 2025`
> (ajuste os anos conforme o escopo definido para o projeto).

In [ ]:
import io
import os
import sys

sys.path.insert(0, os.path.abspath(".."))

import boto3
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from botocore.client import Config

from src.config import MINIO_ACCESS_KEY, MINIO_ENDPOINT, MINIO_SECRET_KEY

sns.set_theme(style="whitegrid")

ANO = 2025
BUCKET_REFINED = os.getenv("MINIO_REFINED_BUCKET", "refined")

s3 = boto3.client(
    "s3",
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
    config=Config(signature_version="s3v4"),
)

key = f"cobertura_vacinal/ano={ANO}/cobertura_municipios.parquet"
obj = s3.get_object(Bucket=BUCKET_REFINED, Key=key)
df = pd.read_parquet(io.BytesIO(obj["Body"].read()))
df.shape

## 1. Visão geral

In [ ]:
df.head()

In [ ]:
df.info()
print("\nValores ausentes por coluna:")
print(df.isna().sum())

In [ ]:
df["cobertura_doses_por_100_habitantes"].describe()

## 2. Distribuição da cobertura vacinal

Lembrete: a métrica é "doses aplicadas por 100 habitantes" (proxy de
intensidade de vacinação), não "% de pessoas vacinadas" — ver
`docs/decisoes_limpeza.md`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df["cobertura_doses_por_100_habitantes"], bins=40, ax=axes[0])
axes[0].set_title("Distribuição — doses por 100 habitantes")

sns.boxplot(x=df["cobertura_doses_por_100_habitantes"], ax=axes[1])
axes[1].set_title("Boxplot — identificação visual de outliers")

plt.tight_layout()
plt.savefig("../reports/distribuicao_cobertura.png", dpi=150)
plt.show()

## 3. Municípios extremos

Menor e maior cobertura — os de maior cobertura merecem checagem: podem ser
polos regionais de vacinação (atendem população de municípios vizinhos), o
que infla o numerador sem refletir a população local (ver coluna
`outlier_iqr` no dataset `trusted` de doses, por mês).

In [ ]:
colunas_exibir = [
    "codigo_municipio",
    "municipio",
    "populacao",
    "doses_aplicadas",
    "cobertura_doses_por_100_habitantes",
]

print("10 municípios com MENOR cobertura:")
display(df.nsmallest(10, "cobertura_doses_por_100_habitantes")[colunas_exibir])

print("\n10 municípios com MAIOR cobertura:")
display(df.nlargest(10, "cobertura_doses_por_100_habitantes")[colunas_exibir])

## 4. Cobertura vs. população

Municípios pequenos tendem a ter métricas mais voláteis (poucas doses já
mudam muito o percentual) — útil para decidir se vale a pena um corte
mínimo de população antes da Etapa 3 (clusterização/classificação).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
sns.scatterplot(
    data=df,
    x="populacao",
    y="cobertura_doses_por_100_habitantes",
    alpha=0.5,
    ax=ax,
)
ax.set_xscale("log")
ax.set_xlabel("População (escala log)")
ax.set_ylabel("Doses por 100 habitantes")
ax.set_title("Cobertura vacinal vs. população do município")
plt.tight_layout()
plt.savefig("../reports/cobertura_vs_populacao.png", dpi=150)
plt.show()

## 5. Próximos passos (pendente de dados adicionais)

Análise de correlação com variáveis socioeconômicas (renda, IDH) — citada
no objetivo do projeto — depende de coletar essas variáveis do IBGE/SIDRA,
o que ainda não foi feito na Etapa 1 (só população foi coletada). Ver
"Pendências conhecidas" em `docs/decisoes_limpeza.md`.